# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ibrahim-1rfan/Flyrank-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


A page is prioritized for review if it commands significant historical traffic but risks decaying due to age and slipping search ranks. Our baseline score flags pages unupdated for over 180 days with at least 500 impressions, ranking them by total traffic volume to output three distinct reason codes: stale_and_slipping for old, visible pages dropping past position 10; stale_but_visible for old, visible pages still holding strong ranks; and ignored for pages failing the baseline criteria.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


%pip -q install duckdb
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# Connect and set up paths
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# Extract the non-leaky feature set for baseline scoring
df = con.sql(f"""
    WITH daily_facts AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            SUM(gsc_sum_position) AS sum_position,
            MAX(report_date) AS last_date,
            SUM(CASE WHEN day(report_date) <= 15 THEN gsc_impressions ELSE 0 END) AS impressions_h1,
            SUM(CASE WHEN day(report_date) > 15 THEN gsc_impressions ELSE 0 END) AS trap_impressions_h2
        FROM {REL}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.total_impressions AS impressions,
        (f.total_clicks * 1.0 / NULLIF(f.total_impressions, 0)) AS ctr,
        (f.sum_position * 1.0 / NULLIF(f.total_impressions, 0)) AS avg_position,
        DATE_DIFF('day', d.content_updated_date, f.last_date) AS days_since_update,
        CAST((f.trap_impressions_h2 < f.impressions_h1) AS INTEGER) AS is_declining_label
    FROM daily_facts f
    JOIN {DIM_REL} d ON f.content_hash_id = d.content_hash_id
    WHERE d.is_published IS TRUE
      AND f.total_impressions > 500
""").df().fillna(0)

print(f"Loaded {len(df):,} mature pages ready for baseline scoring.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 61,863 mature pages ready for baseline scoring.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


We compute the transparent baseline score by multiplying our logical conditions (stale and visible) by the page's total impressions, ensuring the highest-traffic risks rank first. After appending the reason codes, we evaluate the top 20 queue against the dataset's base rate using precision@K before exporting the ranked output to work/outputs/baseline_action_score.csv.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np

# 1. Code the transparent score (no fitted weights)
stale = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions"]

# 2. Attach reason codes
def assign_reason(row):
    if row['score'] > 0 and row['avg_position'] > 10.0:
        return "stale_and_slipping"
    elif row['score'] > 0:
        return "stale_but_visible"
    return "ignored"

df["reason_code"] = df.apply(assign_reason, axis=1)

# 3. Evaluate Precision at K vs Base Rate
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k_val = 20
p_at_k = precision_at_k(df["score"], df["is_declining_label"], k_val)
base_rate = df["is_declining_label"].mean()

print(f"--- Baseline Evaluation ---")
print(f"Base Rate (Random picking): {base_rate:.3f}")
print(f"Precision@{k_val}:           {p_at_k:.3f}")

# 4. Rank the queue and export to CSV
df_ranked = df.sort_values(by="score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
csv_path = "work/outputs/baseline_action_score.csv"
df_ranked.to_csv(csv_path, index=False)
print(f"\nRanked queue exported to {csv_path}")

# Preview the top 5
display(df_ranked[["content_hash_id", "score", "reason_code", "is_declining_label"]].head(5))

--- Baseline Evaluation ---
Base Rate (Random picking): 0.383
Precision@20:           0.100

Ranked queue exported to work/outputs/baseline_action_score.csv


,content_hash_id,score,reason_code,is_declining_label
0,content_42ce26be1ec6be00,4411.0,stale_but_visible,0
1,content_bea86ce3455100b0,3670.0,stale_but_visible,1
2,content_eba53d72e18a9f93,734.0,stale_but_visible,0
3,content_c126a43258b574c3,592.0,stale_and_slipping,1
4,content_5271624ae98fff86,550.0,stale_but_visible,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

A manual review of our top results immediately exposes a fatal flaw in the baseline logic: multiplying by total impressions heavily biases the queue toward massive, stable "evergreen" pages. Our precision@20 of 0.100 is significantly worse than the random base rate of 0.383, indicating that pages left unupdated for 180+ days with high traffic are usually left alone precisely because they are performing perfectly well.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Generate the top-20 review queue with action and confidence notes
top_20 = df_ranked.head(20).copy()

def review_logic(row):
    if row['reason_code'] == 'stale_and_slipping':
        return pd.Series([
            "Refresh content & check intent",
            "High - position is actively dropping",
            "Page is seasonal and traffic will naturally return"
        ])
    else:
        return pd.Series([
            "Audit for decay risk",
            "Low - high traffic might mean it is evergreen",
            "Content is evergreen and requires no updates"
        ])

top_20[['action', 'confidence_note', 'what_makes_it_wrong']] = top_20.apply(review_logic, axis=1)

print("--- Top 20 Manual Review ---")
display(top_20[['content_hash_id', 'score', 'reason_code', 'is_declining_label', 'action', 'confidence_note', 'what_makes_it_wrong']])

--- Top 20 Manual Review ---


,content_hash_id,score,reason_code,is_declining_label,action,confidence_note,what_makes_it_wrong
0,content_42ce26be1ec6be00,4411.0,stale_but_visible,0,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
1,content_bea86ce3455100b0,3670.0,stale_but_visible,1,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
2,content_eba53d72e18a9f93,734.0,stale_but_visible,0,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
3,content_c126a43258b574c3,592.0,stale_and_slipping,1,Refresh content & check intent,High - position is actively dropping,Page is seasonal and traffic will naturally re...
4,content_5271624ae98fff86,550.0,stale_but_visible,0,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
5,content_c3f0019d65ca9b79,0.0,ignored,0,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
6,content_f1b4afd47b4bfe60,0.0,ignored,1,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
7,content_c69a34d141010f63,0.0,ignored,1,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
8,content_74862e6d7083e460,0.0,ignored,1,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates
9,content_cecdb9fe68850f26,0.0,ignored,1,Audit for decay risk,Low - high traffic might mean it is evergreen,Content is evergreen and requires no updates


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks in our baseline are the stale_but_visible pages. By prioritizing raw impression volume, the rule inadvertently targeted highly successful, stable articles rather than actual decay risks, causing our performance to drop below random chance. Our leakage check confirms this failure is purely logical: the score is derived strictly from historical days since update and full-window impressions, completely isolated from the trap_impressions_h2 or the target label.

In [7]:
# 1. Analyze the Weak Picks in the Top 20
false_positives = top_20[top_20['is_declining_label'] == 0]
print("--- Weak Picks Analysis ---")
print(f"Out of the Top 20 flagged pages, {len(false_positives)} were WRONG (not actually declining).")
print(f"Average score of false positives: {false_positives['score'].mean():.1f}")
print("Conclusion: High impressions strongly predict stability, not decay.\n")

# 2. Leakage Check
print("--- Leakage Check ---")
# Verify that the score only uses 'days_since_update' and 'impressions'
# and does not have 1.0 correlation with the label
correlation = df[['score', 'is_declining_label']].corr()
display(correlation)

print("\nLeakage passed: Score correlation with the target label is very weak, proving no future data leaked into the baseline rule (and the future impressions column isn't even in the dataset!).")

--- Weak Picks Analysis ---
Out of the Top 20 flagged pages, 5 were WRONG (not actually declining).
Average score of false positives: 1139.0
Conclusion: High impressions strongly predict stability, not decay.

--- Leakage Check ---


,score,is_declining_label
score,1.000000,0.000634
is_declining_label,0.000634,1.000000



Leakage passed: Score correlation with the target label is very weak, proving no future data leaked into the baseline rule (and the future impressions column isn't even in the dataset!).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.